# 청크 크기에 따른 검색 결과 비교

이 노트북은 `chunk_size`와 `chunk_overlap` 설정에 따라 의미 검색 결과가 어떻게 달라지는지 비교합니다.

| 설정 | chunk_size | chunk_overlap |
|------|-----------|---------------|
| **기본** | 1000 | 200 |
| **소형** | 500 | 100 |
| **초소형** | 300 | 50 |

## 0단계: 환경 설정 및 PDF 로드

In [1]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

load_dotenv()

# PDF 로드
loader = PyPDFLoader("example_data/nke-10k-2023.pdf")
docs = loader.load()
print(f"로드된 페이지 수: {len(docs)}")

# 임베딩 모델
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

로드된 페이지 수: 106


## 1단계: 청크 크기별 분할 및 벡터 저장소 생성

In [2]:
# 청크 설정 정의
chunk_configs = [
    {"name": "기본 (1000/200)", "chunk_size": 1000, "chunk_overlap": 200},
    {"name": "소형 (500/100)", "chunk_size": 500, "chunk_overlap": 100},
    {"name": "초소형 (300/50)", "chunk_size": 300, "chunk_overlap": 50},
]

# 각 설정별로 분할 + 벡터 저장소 생성
vector_stores = {}

for config in chunk_configs:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=config["chunk_size"],
        chunk_overlap=config["chunk_overlap"],
        add_start_index=True,
    )
    splits = splitter.split_documents(docs)
    
    store = InMemoryVectorStore(embeddings)
    store.add_documents(documents=splits)
    
    vector_stores[config["name"]] = store
    print(f"{config['name']}: {len(splits)}개 청크")

기본 (1000/200): 501개 청크
소형 (500/100): 942개 청크
초소형 (300/50): 1543개 청크


## 2단계: 검색 결과 비교

동일한 쿼리를 각 설정의 벡터 저장소에 보내서 결과를 비교합니다.

In [3]:
queries = [
    "How many distribution centers does Nike have in the US?",  # 나이키는 미국에 물류센터가 몇 개인가?
    "When was Nike incorporated?",  # 나이키는 언제 설립되었는가?
    "What was Nike's revenue?",  # 나이키의 매출은 얼마인가?
]

for query in queries:
    print(f"{'=' * 70}")
    print(f"쿼리: {query}")
    print(f"{'=' * 70}")
    
    for name, store in vector_stores.items():
        results = store.similarity_search_with_score(query, k=1)
        doc, score = results[0]
        print(f"\n[{name}] 점수: {score:.4f}")
        print(f"  내용: {doc.page_content[:200]}...")
    
    print()

쿼리: How many distribution centers does Nike have in the US?

[기본 (1000/200)] 점수: 0.7615
  내용: our Greater China geography, occupied by employees focused on implementing our wholesale, NIKE Direct and merchandising 
strategies in the region, among other functions.
In the United States, NIKE has...

[소형 (500/100)] 점수: 0.7426
  내용: our Greater China geography, occupied by employees focused on implementing our wholesale, NIKE Direct and merchandising 
strategies in the region, among other functions.
In the United States, NIKE has...

[초소형 (300/50)] 점수: 0.7581
  내용: TOTAL  369 
In the United States, NIKE has eight significant distribution centers. Refer to Item 2. Properties for further information.
NIKE, INC.       2...

쿼리: When was Nike incorporated?

[기본 (1000/200)] 점수: 0.6119
  내용: NIKE, INC.
One Bowerman Drive
Beaverton, OR 97005-6453
www.nike.com...

[소형 (500/100)] 점수: 0.6120
  내용: NIKE, INC.
One Bowerman Drive
Beaverton, OR 97005-6453
www.nike.com...

[초소형 (300/50)] 점수: 0.6120
  내용:

## 3단계: 정답 포함 여부 확인

각 청크 설정에서 정답 키워드가 포함된 청크가 몇 번째로 검색되는지 확인합니다.

In [4]:
# 쿼리와 기대하는 정답 키워드
query_answer_pairs = [
    ("How many distribution centers does Nike have in the US?", "eight significant distribution centers"),
    ("When was Nike incorporated?", "incorporated in 1967"),
    ("What was Nike's revenue?", "51,217"),
]

for query, answer_keyword in query_answer_pairs:
    print(f"쿼리: {query}")
    print(f"정답 키워드: '{answer_keyword}'")
    
    for name, store in vector_stores.items():
        results = store.similarity_search_with_score(query, k=5)
        found_at = None
        for i, (doc, score) in enumerate(results):
            if answer_keyword.lower() in doc.page_content.lower():
                found_at = i + 1
                break
        
        if found_at:
            print(f"  [{name}] → {found_at}번째 결과에서 발견 (점수: {results[found_at-1][1]:.4f})")
        else:
            print(f"  [{name}] → 상위 5개에 없음")
    
    print()

쿼리: How many distribution centers does Nike have in the US?
정답 키워드: 'eight significant distribution centers'
  [기본 (1000/200)] → 1번째 결과에서 발견 (점수: 0.7615)
  [소형 (500/100)] → 1번째 결과에서 발견 (점수: 0.7426)
  [초소형 (300/50)] → 1번째 결과에서 발견 (점수: 0.7580)

쿼리: When was Nike incorporated?
정답 키워드: 'incorporated in 1967'
  [기본 (1000/200)] → 3번째 결과에서 발견 (점수: 0.5521)
  [소형 (500/100)] → 2번째 결과에서 발견 (점수: 0.5875)
  [초소형 (300/50)] → 2번째 결과에서 발견 (점수: 0.6095)

쿼리: What was Nike's revenue?
정답 키워드: '51,217'
  [기본 (1000/200)] → 1번째 결과에서 발견 (점수: 0.6194)
  [소형 (500/100)] → 1번째 결과에서 발견 (점수: 0.6505)
  [초소형 (300/50)] → 2번째 결과에서 발견 (점수: 0.6533)

